# 六列表详细一致性检查

本 Notebook 不运行信号引擎，也不读取远端现货。它只读取：

- 第一个 Notebook 生成的 `runtime_outputs/最终执行日简表.csv`；
- 包内 `expected/local_freeze/最终执行日简表_本地冻结参考.csv`。

它会逐日、逐列检查实际执行日、三状态、两个非零反转和大涨大跌五类信号，并输出详细的机器可读结论、逐日对比文件和不一致示例。

In [ ]:
from pathlib import Path
import sys

def find_package_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'compare_compact_output.py').is_file():
            return candidate
    raise FileNotFoundError('没有找到最终六列运行包根目录。请确认 Notebook 与 src、expected 位于同一目录。')

PACKAGE_ROOT = find_package_root(Path.cwd().resolve())
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from compare_compact_output import compare_compact_output

conclusion = compare_compact_output()
conclusion

In [ ]:
import json
import pandas as pd

conclusion_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表一致性结论.json'
comparison_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表逐日对比.csv'
sample_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表不一致示例.csv'
conclusion = json.loads(conclusion_path.read_text(encoding='utf-8'))
comparison = pd.read_csv(comparison_path, encoding='utf-8-sig')

print('===== 最终检查结论 =====')
print('success =', conclusion['success'])
print('日期和五类信号全部一致 =', conclusion['all_dates_and_signals_match'])
print('远端日期：', conclusion['remote_date_min'], '->', conclusion['remote_date_max'])
print('本地日期：', conclusion['local_date_min'], '->', conclusion['local_date_max'])
print('远端行数：', conclusion['remote_rows'], '本地行数：', conclusion['local_rows'])

field_rows = []
for field, metrics in conclusion['field_metrics'].items():
    field_rows.append({
        '字段': field,
        '共同日期不一致行数': metrics['mismatch_rows_on_common_dates'],
        '共同日期一致行数': metrics['match_rows_on_common_dates'],
    })
print('===== 按字段详细检查 =====')
display(pd.DataFrame(field_rows))

print('===== 前 50 条不一致或日期差异 =====')
sample = pd.read_csv(sample_path, encoding='utf-8-sig')
display(sample)

print('===== 对比文件列 =====')
print(list(comparison.columns))
print('说明：success=True 才表示六列表的日期、状态和全部信号值完全一致。')